# Training a phase reconstructor for Ekarus

This notebook trains a convolutional neural network to reconstruct wavefronts from Ekarus's pyramid WFS detector frames, using the fully differentiable end-to-end simulation.

Unlike the calibration notebook — where we fit the WFS and DM to match real bench measurements — here the WFS and DM are loaded from a previous calibration and then frozen (`requires_grad_(False)`); only the reconstructor network's weights are updated.

Training simulates an AO closed loop: at each step we draw a random turbulent phase screen, propagate the *residual* phase (after applying the previous correction) through the WFS, and backpropagate the reconstruction error through the optical model and DM straight into the network. This is implemented by `AI4AO.Trainer.Trainer` (see `AI4AO/Trainer.py`), which this notebook uses for training, evaluation, loss plotting, and checkpointing — replacing the hand-written closed-loop training loop this notebook used to have.

In [ ]:
from mmengine import Config
import matplotlib.pyplot as plt
import torch
import numpy as np
import torch.nn as nn

from AI4AO import PyramidWFS, PhaseDataset, FramePreprocess, DeformableMirror, Trainer, imshow, imshow_multiple
from AI4AO.LossFunctions import LogResidualVarianceLoss

## Reconstructor architecture

`PWFSNet` maps the 4 preprocessed pyramid pupil images to a vector of `Nmodes` KL-mode coefficients. Unlike the simpler single-head CNNs used for the other instruments, this one splits the 768 encoder output channels into 3 groups of 256 and gives the low/mid/high mode ranges (`num_low`/`num_mid`/`num_high`) their own linear head each — useful since low-order modes (tip/tilt, focus, ...) and high-order modes typically need different amounts of spatial context to estimate well. The `shuffler` interleaves the 3 grouped-convolution branches from the stem before that split, so each head still sees a mix of all 3 branches rather than being locked to whichever branch happened to land in its slice.

(The notebook used to also define a `SecondaryNetwork` MLP meant to post-process `PWFSNet`'s output, but it was never actually wired into the training loop — dropped here as dead code.)

In [ ]:
class PWFSNet(nn.Module):
    def __init__(self, DMParams):
        super().__init__()

        Nmodes = DMParams["Nmodes"]

        self.num_low = 50
        self.num_mid = 100
        self.num_high = Nmodes - (self.num_low + self.num_mid)

        self.stem = nn.Sequential(
            # Process each pupil independently
            nn.Conv2d(4, 48, kernel_size=11, padding=5, groups=4),
            nn.LeakyReLU(),

            nn.Conv2d(48, 96, kernel_size=7, padding=3, groups=12),
            nn.LeakyReLU(),

            nn.AvgPool2d(2),      # 42 -> 21
        )

        self.encoder = nn.Sequential(
            nn.Conv2d(96, 192, 5, padding=2, groups=3),
            nn.LeakyReLU(),

            nn.Conv2d(192, 192, 5, padding=2, groups=3),
            nn.LeakyReLU(),

            nn.AvgPool2d(2),      # 21 -> 10

            nn.Conv2d(192, 384, 3, padding=1, groups=3),
            nn.LeakyReLU(),

            nn.Conv2d(384, 384, 3, padding=1, groups=3),
            nn.LeakyReLU(),

            nn.AvgPool2d(2),      # 10 -> 5

            nn.Conv2d(384, 768, 3, padding=1, groups=3),
            nn.LeakyReLU(),

            nn.AdaptiveAvgPool2d(1)
        )

        self.flatten = nn.Flatten()
        self.head_low = nn.Linear(256, self.num_low, bias=False)
        self.head_mid = nn.Linear(256, self.num_mid, bias=False)
        self.head_high = nn.Linear(256, self.num_high, bias=False)

        self.apply(self._init_weights)

        # The 3 groups above each see a mix of the 3 head slices; this shuffle
        # interleaves channels across groups so each head draws on all of them.
        shuffler = torch.arange(96, device=device)
        self.shuffler = shuffler.view(-1, 3, 8).permute(1, 0, 2).flatten()

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.kaiming_normal_(module.weight, nonlinearity="relu")
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, x):
        x = self.stem(x)
        x = x[:, self.shuffler]
        x = self.encoder(x)
        low_features, mid_features, high_features = torch.chunk(x, chunks=3, dim=1)
        out_low = self.head_low(self.flatten(low_features))
        out_mid = self.head_mid(self.flatten(mid_features))
        out_high = self.head_high(self.flatten(high_features))
        return torch.cat((out_low, out_mid, out_high), dim=1)

## Loading the instrument configuration

Each instrument has a dedicated params file (here `Ekarus_params.py`) holding the plain-dict configs consumed positionally by the pipeline constructors — `WFSParams`, `AtmosParams`, `LoopParams`, `TrainParams`, `DMParams`. There is no YAML/JSON config layer in this codebase; these dicts *are* the configuration mechanism.

In [ ]:
device = 'cuda' # set to "cpu" if Cuda is not available

paramfile = 'Ekarus_params.py'  # file of experimental parameters

# Config extraction
AtmosParams = Config.fromfile(paramfile)['AtmosParams']
WFSParams = Config.fromfile(paramfile)['WFSParams']
LoopParams = Config.fromfile(paramfile)['LoopParams']
TrainParams = Config.fromfile(paramfile)['TrainParams']
DMParams = Config.fromfile(paramfile)['DMParams']

## Dataset

`PhaseDataset` generates atmospheric phase screens on the fly from a von Kármán PSD. Setting `dataset.generateClosedLoop = True` selects the closed-loop (AO-residual) PSD instead of raw open-loop turbulence — see `Tutorials/basics/01_Dataset.ipynb` for exactly what that does and doesn't change.

In [ ]:
# Dataset creation
dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
dataset.generateClosedLoop = True

## Loading the calibrated WFS and DM

We load the WFS mask geometry and DM misregistration/influence functions fit in `CalibrateExampleEkarusTwin.ipynb`, then freeze them (`.eval()` + `requires_grad_(False)`) — in this notebook only the reconstructor network is trained, so gradients should not flow into the optical model or DM geometry. Re-centering the influence functions (masking by the pupil, then subtracting each actuator's mean over the pupil) removes DC offsets outside the aperture so DM commands don't inject unphysical piston into the reconstructed phase.

`PATH` uses `../../Data/Ekarus/` (not `../Data/Ekarus/`) since this notebook now lives in `Tutorials/Ekarus/`, one level deeper than `Tutorials/` — the old single-`..` path is stale after that move and would silently point at a nonexistent folder.

In [ ]:
PATH = "../../Data/Ekarus/"

# WFS creation
wfs = PyramidWFS(WFSParams, device)
wfs.LoadCalibration(PATH + "EkarusWFS.pth")
wfs.eval()

dm = DeformableMirror(WFSParams, DMParams, device)
dm.LoadCalibration(PATH + "EkarusDM.pth")
dm.eval()

## Modes-to-commands matrix

`M2C` converts a vector of modal coefficients into DM actuator commands, so `dm(M2C.T)` gives the full-resolution phase produced by each individual mode on its own. `z_inv`, its pseudo-inverse, does the reverse: projecting a full-resolution phase screen onto modal coefficients. This is how the training loop obtains the "ground truth" modal coefficients for a given turbulence phase screen, to compare against the reconstructor's prediction.

`M2C_KL_OOPAO.npy` is a KL basis computed in OOPAO from the same bench calibration as the interaction matrix used later in this notebook (`IM.npy`) — the two share a mode basis and ordering.

In [ ]:
M2C = np.load(PATH + "M2C_KL_OOPAO.npy")
M2C = torch.from_numpy(M2C).to(device=device, dtype=torch.float32)
M2C = M2C[:, :DMParams["Nmodes"]]  # keep only the modes this reconstructor is trained to output

z_inv = torch.linalg.pinv(dm(M2C.T).flatten(start_dim=-2))  # full-resolution phase -> modal coefficients

## Frame preprocessing

`FramePreprocess` crops the individual pupil images out of the raw WFS detector frame and reference-subtracts/normalizes them before they reach the reconstructor. `ProcessReference` records the WFS's own flat-wavefront reference intensity, which is subtracted from every subsequent frame passed through `ProcessFrame`.

In [ ]:
# frame processor creation
framePreprocessor = FramePreprocess(WFSParams, wfs, device)
framePreprocessor.ProcessReference(wfs.reference_intensity)

## Instantiating the reconstructor

We build the network and (if present) resume from a previously trained checkpoint further below, once the `Trainer` exists — see the "The Trainer" section.

In [ ]:
# Phase reconstructor
phaseReconstructor = PWFSNet(DMParams).to(device=device)

total_params = sum(p.numel() for p in phaseReconstructor.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")

## Optimizer and loss

We optimize only the reconstructor's parameters, with AdamW. `LogResidualVarianceLoss` is a physics-aware loss: it computes `ln(var(residual phase over the pupil))`, i.e. the log-variance of the wavefront error in radians², which is directly tied to the residual RMS/Strehl ratio the AO loop would achieve — not just an abstract regression loss on the mode coefficients.

In [ ]:
# Optimization parameters (learning rate lr and nb of runs)
lrn = TrainParams['lrn']
num_iterations = 1 # Closed loop iterations

optimizer_n = torch.optim.AdamW(phaseReconstructor.parameters(), lrn, fused=True)

# Setting the loss
loss_variance = LogResidualVarianceLoss(dataset.pupil)

Re-run the cell below with a different `TrainRunNb`/`num_iterations` to control how many optimizer steps `trainer.train()` performs, without re-creating (and losing the momentum state of) the optimizer defined above.

In [ ]:
TrainRunNb = 5000
num_iterations = 1

## The Trainer

`AI4AO.Trainer.Trainer` bundles the WFS, DM, frame preprocessor, modal basis (`M2C`), reconstructor, dataset, loss and optimizer, and implements the closed-loop training step (see `AI4AO/Trainer.py`). It also exposes `save_checkpoint`/`load_checkpoint` for persisting reconstructor + optimizer state, and the `evaluate`/`plot_losses` helpers used further down.

We try to resume from a previous checkpoint before training further. Note: an `EkarusCNN.pth` saved by the older `torch.save(phaseReconstructor.state_dict(), ...)` pattern is a raw state dict, not the wrapped format `save_checkpoint` writes, so loading it here will raise `KeyError` and fall back to training from scratch — re-save once with `trainer.save_checkpoint(...)` to make it loadable by `load_checkpoint` going forward.

In [ ]:
trainer = Trainer(wfs=wfs,
                  framePreprocessor=framePreprocessor,
                  dm=dm,
                  M2C=M2C,
                  phaseReconstructor=phaseReconstructor,
                  dataset=dataset,
                  loss=loss_variance,
                  optimizer=optimizer_n)

try:
    trainer.load_checkpoint(PATH + "EkarusCNN.pth", load_optimizer=False)
except KeyError:
    # Existing checkpoint predates save_checkpoint's format (a raw state_dict);
    # once re-saved with trainer.save_checkpoint it will load cleanly here.
    print("Starting from scratch")

## Training

`trainer.train(training_steps, closed_loop_iterations)` runs `training_steps` closed-loop optimizer updates. `closed_loop_iterations` sets how many AO-loop steps are simulated — and backpropagated through — per optimizer update; with more than 1, the reconstructor is trained to perform well *given* its own previous corrections, rather than only on independent open-loop frames.

It returns two per-step loss trackers: `loss_tracker`, the network's actual training loss, and `loss_tracker_ideal`, the loss that would result from a perfect projection of the true residual phase onto the modal basis instead of the network's prediction — a lower bound to compare against.

In [ ]:
loss_tracker, loss_tracker_ideal = trainer.train(TrainRunNb, 10)

`trainer.plot_losses` smooths and plots both trackers together. The gap between the training loss and the ideal-loss lower bound indicates how much reconstruction performance is still on the table for the network to gain, versus how much is fundamental to the chosen modal basis and WFS.

In [ ]:
trainer.plot_losses(loss_tracker, loss_tracker_ideal)

## Saving

Persist the trained reconstructor, along with the optimizer state (for resuming later), to disk via `trainer.save_checkpoint`.

In [ ]:
trainer.save_checkpoint(PATH + "EkarusCNN.pth")

## Visualizing a closed loop

`trainer.evaluate()` runs a no-grad closed-loop rollout (reconstructor in `.eval()` mode, no pupil noise injected) and returns an `EvaluationResult` holding the phase, pupil, reconstructed phase, residual phase and WFS frames at every simulated step, ready to animate.

In [ ]:
import matplotlib as mpl
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

n_frames = 100
result = trainer.evaluate(n_steps=n_frames, dataset=dataset)

fig, axes = imshow_multiple(
    [result.phase[0], result.residual_phase[0], result.wfs_frames[0], result.psfs[0]],
    same_scale=True,
    titles=["Input phase", "Residual phase", "WFS frame", "PSF"],
    max_channel_number = 4
)


def update(i):
    imshow_multiple(
        [result.phase[i], result.residual_phase[i], result.wfs_frames[i], result.psfs[i]],
        fig=fig, axes=axes, 
        same_scale=True,
        max_channel_number = 4
    )
    return [ax.images[0] for tensor_axes in axes for ax in tensor_axes]


anim = FuncAnimation(fig, update, frames=n_frames, interval=50, blit=False)
plt.close(fig)

mpl.rcParams["animation.embed_limit"] = 100
HTML(anim.to_jshtml())

### With scintillation

Same rollout, but on a fresh dataset with `AtmosParams['Scintillation'] = True` — amplitude fluctuations from angular-spectrum propagation are added on top of the phase screens, which the reconstructor was not trained to handle. Useful to see how it degrades outside its training distribution.

In [ ]:
scint_AtmosParams = AtmosParams.copy()
scint_AtmosParams['Scintillation'] = True
scint_dataset = PhaseDataset(WFSParams, scint_AtmosParams, LoopParams, DMParams, device)
scint_dataset.generateClosedLoop = True

result = trainer.evaluate(n_steps=n_frames, dataset=scint_dataset)

fig, axes = imshow_multiple(
    [result.pupil[0], result.phase[0], result.residual_phase[0], result.wfs_frames[0], result.psfs[0]],
    same_scale=True,
    titles=["Pupil amplitude", "Input phase", "Residual phase", "WFS frame", "PSF"],
    max_channel_number = 4
)


def update(i):
    imshow_multiple(
        [result.pupil[i], result.phase[i], result.residual_phase[i], result.wfs_frames[i], result.psfs[i]],
        fig=fig, axes=axes, 
        same_scale=True,
        max_channel_number = 4
    )
    return [ax.images[0] for tensor_axes in axes for ax in tensor_axes]


anim = FuncAnimation(fig, update, frames=n_frames, interval=50, blit=True)
plt.close(fig)

mpl.rcParams["animation.embed_limit"] = 100
HTML(anim.to_jshtml())

## Sanity check against the real bench interaction matrix

Everything above validates the network only against the *simulator*. As an independent check, `Data/Ekarus/IM.npy` is a real interaction matrix measured on the Ekarus bench (in the same KL basis as `M2C_KL_OOPAO.npy`, so its reconstructed coefficients plug directly into `dm(... @ M2C.T)`), and `valid_pix_map.npy` records which detector pixels it was measured over. Inverting it gives a classical linear reconstructor built entirely from hardware measurements — no simulation assumptions at all — that we can compare the trained CNN against on a *simulated* frame.

This crops the detector frame the same way the original bench acquisition did (`iMat[:, 40:-60, 50:-50]`); if the WFS frame resolution/cropping changes upstream, this crop needs revisiting.

In [ ]:
bench_iMat = np.load(PATH + "IM.npy").T
bench_iMat = torch.from_numpy(bench_iMat.astype(np.float32)).to(device=device, dtype=torch.float32)

nModes_bench = bench_iMat.shape[0]

valid_pix = np.load(PATH + "valid_pix_map.npy").astype(np.bool_)
W, H = valid_pix.shape
nPix = int(valid_pix.sum())

bench_iMat = bench_iMat[:, :nPix]

iMat = torch.zeros((nModes_bench, W, H), device=device, dtype=torch.float32)
iMat[:, valid_pix] = bench_iMat
iMat = iMat[:, 40:-60, 50:-50]

bench_recon = torch.linalg.pinv(iMat.flatten(start_dim=-2))
M2C_bench = M2C[:, :nModes_bench]

In [ ]:
phaseReconstructor.eval()

batch = dataset[0]
phaseGT = batch["phase"]
wfs.SetPhotonsAndRON(batch["nphotons"], batch["ron"])

with torch.no_grad():
    wfs_frame = wfs(phaseGT, batch["pupil"])
    preprocessed_frames = framePreprocessor.ProcessFrame(wfs_frame, False)
    z_cnn = phaseReconstructor(preprocessed_frames)
    phase_cnn = dm(z_cnn @ M2C.T)

    z_bench = wfs_frame.flatten(start_dim=-2) @ bench_recon
    phase_bench = dm(z_bench @ M2C_bench.T)

imshow_multiple(
    [phaseGT, phase_cnn, phase_bench, phaseGT - phase_cnn, phaseGT - phase_bench],
    same_scale=False
)
plt.show()

print(f"CNN residual variance:            {torch.var((phaseGT - phase_cnn)[:, dataset.pupil], dim=-1).mean().item():.4f} rad^2")
print(f"Bench linear recon residual var.: {torch.var((phaseGT - phase_bench)[:, dataset.pupil], dim=-1).mean().item():.4f} rad^2")